# Import libraries and datasets

In [ ]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer

from utils.analysis_functions import find_differences, get_columns_with_missing_values
from utils.data_transformation_functions import transform_transaction_datetime, transform_transactionAmt, replace_card, group_email_domains
from utils.data_encoding_functions import one_hot_encode_columns, frequency_encoding
from utils.data_inputation_functions import DataImputer

In [ ]:
train_identity_df = pd.read_csv('../data/train_identity.csv')
train_transaction_df = pd.read_csv('../data/train_transaction.csv')
test_identity_df = pd.read_csv('../data/test_identity.csv')
test_transaction_df = pd.read_csv('../data/test_transaction.csv')

In [ ]:
train_data_df = train_transaction_df.merge(train_identity_df, how='left', on='TransactionID')
test_data_df = test_transaction_df.merge(test_identity_df, how='left', on='TransactionID')

del train_identity_df, train_transaction_df, test_identity_df, test_transaction_df

# Handle differences in column names

In [ ]:
test_data_df.columns = test_data_df.columns.str.replace('^id-', 'id_', regex=True)
find_differences(train_data_df, test_data_df)

## Handle columns with many missing values

In [ ]:
columns_with_missing_data = get_columns_with_missing_values(train_data_df)

In [ ]:
train_data_df = train_data_df.drop(columns_with_missing_data, axis=1)
test_data_df = test_data_df.drop(columns_with_missing_data, axis=1)

In [ ]:
print("Train data shape:", train_data_df.shape)
print("Test data shape:", test_data_df.shape)

## Handle transactionDT columns

As we explored, we can create new features such as days of the week and hours of the day `TransactionDT` column. 

In [ ]:
train_data_df = transform_transaction_datetime(train_data_df)
test_data_df = transform_transaction_datetime(test_data_df)

In [ ]:
train_data_df

As we have seen in the exploration notebook, adding days of the week and hours of the day as features can help improve the model's performance.

## Handle transactionAmt columns

In [ ]:
train_data_df = transform_transactionAmt(train_data_df)
test_data_df = transform_transactionAmt(test_data_df, isTest=True)

In [ ]:
train_data_df

Employing the logaritmic transformation on the `TransactionAmt` column can help improve the model's performance, since in this way we can reduce the skewness of the data and approach a normal distribution.

## Handle ProductCD columns

In [ ]:
train_data_df = one_hot_encode_columns(train_data_df, ['ProductCD'])
test_data_df = one_hot_encode_columns(test_data_df, ['ProductCD'])

In [ ]:
train_data_df

In [ ]:
train_data_df[['ProductCD_H', 'ProductCD_R', 'ProductCD_S', 'ProductCD_W']].info()

## Handle card columns

In [ ]:
card_columns = [col for col in train_data_df.columns if col.startswith('card')]

numerical_card_columns = [col for col in card_columns if pd.api.types.is_numeric_dtype(train_data_df[col])]
categorical_card_columns = [col for col in card_columns if not pd.api.types.is_numeric_dtype(train_data_df[col])]

In [ ]:
data_imputer = DataImputer(strategy='mean')

train_data_df = data_imputer.impute_train_set_values(train_data_df, numerical_card_columns)
test_data_df = data_imputer.impute_test_set_values(test_data_df, numerical_card_columns)

In [ ]:
data_imputer = DataImputer(strategy='most_frequent')

train_data_df = data_imputer.impute_train_set_values(train_data_df, categorical_card_columns)
test_data_df = data_imputer.impute_test_set_values(test_data_df, categorical_card_columns)

In [ ]:
train_data_df['card6'] = train_data_df['card6'].apply(replace_card)
test_data_df['card6'] = test_data_df['card6'].apply(replace_card)

In [ ]:
train_data_df = one_hot_encode_columns(train_data_df, ['card4', 'card6'])
test_data_df = one_hot_encode_columns(test_data_df, ['card4', 'card6'])

In [ ]:
train_data_df

In [ ]:
train_data_df[['card1', 'card2', 'card3', 'card5', 'card4_discover',	'card4_mastercard', 'card4_visa', 'card6_debit']].info()

## Handle addr columns

In [ ]:
data_imputer = DataImputer(strategy='mean')

train_data_df = data_imputer.impute_train_set_values(train_data_df, ['addr1', 'addr2'])
test_data_df = data_imputer.impute_test_set_values(test_data_df, ['addr1', 'addr2'])

In [ ]:
train_data_df

In [ ]:
train_data_df[['addr1', 'addr2']].info()

## Handle P_emaildomain columns

In [ ]:
data_imputer = DataImputer(strategy='most_frequent')

train_data_df = data_imputer.impute_train_set_values(train_data_df, ['P_emaildomain'])
test_data_df = data_imputer.impute_test_set_values(test_data_df, ['P_emaildomain'])

In [ ]:
train_data_df = group_email_domains(train_data_df, 'P_emaildomain')
test_data_df = group_email_domains(test_data_df, 'P_emaildomain')

In [ ]:
train_data_df['P_emaildomain'].value_counts()

In [ ]:
train_data_df = frequency_encoding(train_data_df, 'P_emaildomain')
test_data_df = frequency_encoding(test_data_df, 'P_emaildomain')

In [ ]:
train_data_df

In [ ]:
train_data_df['P_emaildomain_freq'].info()

## Handle C1-C14 columns 

In [ ]:
c_cols = ['C' + str(i) for i in range(1, 15)]

train_data_df[c_cols].info()

## Handle D1-D15 columns

In [ ]:
d_cols = ['D1', 'D4', 'D10', 'D15']

In [ ]:
data_imputer = DataImputer(strategy='mean')

train_data_df = data_imputer.impute_train_set_values(train_data_df, d_cols)
test_data_df = data_imputer.impute_test_set_values(test_data_df, d_cols)

In [ ]:
train_data_df[d_cols]

In [ ]:
train_data_df[d_cols].info()

## Handle M1-M9 columns

In [ ]:
m_cols = ['M6']

In [ ]:
data_imputer = DataImputer(strategy='most_frequent')

train_data_df = data_imputer.impute_train_set_values(train_data_df, m_cols)
test_data_df = data_imputer.impute_test_set_values(test_data_df, m_cols)

In [ ]:
train_data_df = one_hot_encode_columns(train_data_df, ['M6'])
test_data_df = one_hot_encode_columns(test_data_df, ['M6'])

In [ ]:
train_data_df

In [ ]:
train_data_df['M6_T'].info()

## V1-V339 columns

In [ ]:
v_cols = [
    'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 
    'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 
    'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 
    'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 
    'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 
    'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 
    'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 
    'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V88', 'V89', 'V90', 
    'V91', 'V92', 'V93', 'V94', 'V95', 'V96', 'V97', 'V98', 'V99', 'V100', 
    'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108', 'V109', 'V110', 
    'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V120', 
    'V121', 'V122', 'V123', 'V124', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 
    'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 
    'V279', 'V280', 'V281', 'V282', 'V283', 'V284', 'V285', 'V286', 'V287', 'V288', 'V289', 
    'V290', 'V291', 'V292', 'V293', 'V294', 'V295', 'V296', 'V297', 'V298', 'V299', 
    'V300', 'V301', 'V302', 'V303', 'V304', 'V305', 'V306', 'V307', 'V308', 'V309', 
    'V310', 'V311', 'V312', 'V313', 'V314', 'V315', 'V316', 'V317', 'V318', 'V319', 
    'V320', 'V321'
]

In [ ]:
data_imputer = DataImputer(strategy='mean')

train_data_df = data_imputer.impute_train_set_values(train_data_df, v_cols)
test_data_df = data_imputer.impute_test_set_values(test_data_df, v_cols)

In [ ]:
train_data_df[v_cols].info()

# Review final dataset

In [ ]:
columns_with_missing_data_train = get_columns_with_missing_values(train_data_df)
columns_with_missing_data_test = get_columns_with_missing_values(test_data_df)

print("Columns with missing data in train dataset:", columns_with_missing_data_train)
print("Columns with missing data in test dataset:", columns_with_missing_data_test)

In [ ]:
d_types_train = set(train_data_df.dtypes)
d_types_test = set(test_data_df.dtypes)

print("Data types in train dataset:", d_types_train)
print("Data types in test dataset:", d_types_test)

We will drop `TransactionID` from the training dataset since it is an identifier and does not provide any information to the model. However, we will keep it in the test dataset to create the submission file.

In [ ]:
train_data_df = train_data_df.drop(['TransactionID'], axis=1)

In [ ]:
find_differences(train_data_df, test_data_df)

In [ ]:
train_data_df

In [ ]:
test_data_df

In [ ]:
train_data_df.to_csv('../data/train_data_processed.csv', index=False)
test_data_df.to_csv('../data/test_data_processed.csv', index=False)